# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and explore the FAIR^2 clinical oncology dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
- Croissant schema (JSON-LD): [`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)
- Data: A tabular resource describing 77 cancer survivors with 2nd primary colorectal cancer.

In [ ]:
# Install the mlcroissant library (needed for annotation-based data loading)
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata (access attributes, not as a dict)
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Citation: {getattr(dataset.metadata, 'citeAs', None)}")

## 2. Data Overview

Explore available record sets, their fields, and get their Croissant `@id`s for inspection and loading.

In [ ]:
# List all Record Sets presented in the dataset schema
if hasattr(dataset.metadata, "recordSets") and dataset.metadata.recordSets:
    record_sets = dataset.metadata.recordSets
else:
    # Alternatively, use the 'recordSet' key if present
    record_sets = getattr(dataset.metadata, "recordSet", [])

print("Available record sets:")
for rec in record_sets:
    print(f" - RecordSet name: {getattr(rec, 'name', None)} (@id: {rec.id})")

# If there are no record sets at the package root, load from the table resources
if not record_sets or len(record_sets) == 0:
    print("No explicit record sets defined in the metadata; listing distributions.")
    if hasattr(dataset.metadata, "distribution"):
        for dist in dataset.metadata.distribution:
            print(f" - Distribution @id: {dist.id}; encodingFormat: {getattr(dist, 'encodingFormat', 'N/A')}")
    print("Will attempt loading main table via mlcroissant.record_set_ids.")

# List available record set IDs from the loader
main_record_set_ids = dataset.record_set_ids()
print("\nRecord set IDs found for loading:")
for rid in main_record_set_ids:
    print(rid)
    # Inspect each record set's fields
    fields = dataset.fields(record_set=rid)
    for f in fields:
        print(f"   - Field: {f['@id']}, name: {f.get('name')}, dataType: {f.get('dataType')}")

## 3. Data Extraction

Load all tables/record sets into Pandas dataframes. Use record set, field, and column `@id`s (Croissant IDs).

In [ ]:
# Load data for each available record set using its @id
record_set_ids = dataset.record_set_ids()
dataframes = {}

for rs_id in record_set_ids:
    recs = list(dataset.records(record_set=rs_id))
    if len(recs) > 0:
        dataframes[rs_id] = pd.DataFrame(recs)
        print(f"Loaded dataframe for RecordSet: {rs_id} (records: {len(dataframes[rs_id])})")
        print(f"Columns: {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head())

# Pick a main record set (dataset has only one major tabular set)
main_table_id = record_set_ids[0]  # e.g. 'http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7862866/_/62935be8-24c5-4111-9be6-0b2e3d9593bd'
# Show the field @ids for this table
print(f"\nColumns in record set {main_table_id}:")
for c in dataframes[main_table_id].columns:
    print(f"- {c}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing such as filtering, normalization, and grouping using Croissant field/column `@id`s. Here we select numeric clinical variables for demonstration (e.g., patient age) and filter, normalize, and group.

In [ ]:
# Display all columns so we can pick field IDs for demo
df = dataframes[main_table_id]
print("All columns (Croissant @id):", df.columns.values)

# Let's try to pick a likely age field by @id
age_col = None
for col in df.columns:
    if "age" in col.lower():
        age_col = col
        break
if age_col is None:
    # Default to first numeric column
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        age_col = numeric_candidates[0]
print(f"Selected numeric field for demo (age_col): {age_col}")

if age_col is not None:
    # Ensure all data are numeric, if needed
    df[age_col] = pd.to_numeric(df[age_col], errors='coerce')

    # Filter for patients older than some threshold
    threshold = 55
    filtered_df = df[df[age_col] > threshold]
    print(f"Records with {age_col} > {threshold} (n={len(filtered_df)}):")
    print(filtered_df[[age_col]].head())

    # Normalize the age field
    norm_col = f"{age_col}_normalized"
    filtered_df[norm_col] = (filtered_df[age_col] - filtered_df[age_col].mean()) / filtered_df[age_col].std()
    print(f"\nNormalized {age_col} (z-score):")
    print(filtered_df[[age_col, norm_col]].head())

    # Try grouping by a categorical field (sex, histopathology, etc)
    group_field = None
    for cat_col in df.columns:
        if "sex" in cat_col.lower():
            group_field = cat_col
            break

    if group_field is not None:
        group_stats = filtered_df.groupby(group_field)[age_col].agg(['count','mean','std','min','max'])
        print(f"\nStatistics by group '{group_field}':")
        print(group_stats)
    else:
        print("No suitable group field containing 'sex' found for grouping.")
else:
    print("No numeric age field detected for EDA demonstration.")

## 5. Visualization

Visualize numeric field distributions and group comparisons from the dataset (histogram for age, group mean plot by sex).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the same fields as above
if age_col is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[age_col].dropna(), bins=12, kde=True)
    plt.title(f"Distribution of {age_col}")
    plt.xlabel(f"{age_col}")
    plt.ylabel("Count")
    plt.show()

    # If grouping field available, plot group means with error bars
    if group_field is not None:
        mean_by_group = df.groupby(group_field)[age_col].mean()
        std_by_group = df.groupby(group_field)[age_col].std()
        plt.figure(figsize=(6,4))
        mean_by_group.plot(kind='bar', yerr=std_by_group, capsize=4)
        plt.title(f"Mean {age_col} by {group_field}")
        plt.ylabel(f"Mean {age_col}")
        plt.show()

## 6. Conclusion

- The dataset contains clinical and molecular variables on second primary colorectal cancers in cancer survivors, with rich annotation for comorbidities, anatomical, and treatment features.
- Using `mlcroissant`, we loaded the data and performed initial exploration by Croissant `@id` for transparent schema-tracing analysis.
- We filtered and visualized core numeric variables to demonstrate reusability for further biomarker or phenotyping studies.

See the detailed Croissant schema for full field and record set identifiers when reusing or referencing in downstream pipelines.